In [4]:
from pathlib import Path
import pandas as pd

In [51]:
# detect csv files in all child directories
csv_files = list(Path('../../../data/dataset_sadness_videos/videos').rglob('*.csv'))
csv_files.sort()

In [52]:
len(csv_files)

2

In [53]:
dfs = []
import pandas as pd
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs.append(df)

In [54]:
df = pd.concat(dfs, ignore_index=True)

In [55]:
df.head()

,Segment_id,Start_time_seconds,End_time_seconds,Start_time_formatted,End_time_formatted,Duration_seconds,Sentence
0,0,0.01,1.67,00:00,00:01,1.66,Dzień dobry Państwu.
1,1,1.67,5.31,00:01,00:05,3.64,Miło mi Państwa powitać w kolejnym podcaście Ż...
2,2,5.31,12.11,00:05,00:12,6.80,"Zanim zacznie się rozmowa, chciałbym Państwa w..."
3,3,12.11,14.91,00:12,00:14,2.80,Nie zajmie to Państwu więcej niż kilka sekund.
4,4,14.91,16.99,00:14,00:16,2.08,Życzę dobrego odsłuchu.


In [56]:
len(df)

2557

In [57]:
df.to_csv("../../../data/dataset_sadness_videos/concat.csv", index=False)

In [9]:
df_base = pd.read_csv("../../../data/dataset/concat_annotated.csv")
df_disgust = pd.read_csv("../../../data/dataset_disgusting_videos/concat_annotated_final.csv")
df_scary = pd.read_csv("../../../data/dataset_scary_videos/concat_annotated_scary.csv")
df_anger = pd.read_csv("../../../data/dataset_anger_videos/concat_annotated_anger_enhanced.csv")
df_happy = pd.read_csv("../../../data/dataset_happy_videos/concat_annotated_happiness_enhanced.csv")
df_sadness = pd.read_csv("../../../data/dataset_sadness_videos/concat_annotated_sadness_enhanced.csv")

# concatenate all dataframes
df = pd.concat([df_base, df_disgust, df_scary, df_anger, df_happy, df_sadness], ignore_index=True)
df["Emotion_core"].value_counts()


Emotion_core
neutral      15900
surprise      4294
happiness     1804
disgust       1448
sadness       1057
anger          928
fear           390
Name: count, dtype: int64

In [10]:
# neutral capped at 25% rest between 12 and 17% show desired distribution
distribution = {
    "neutral": 0.20,
    "happiness": 0.165,
    "surprise": 0.165,
    "anger": 0.15,
    "sadness": 0.12,
    "fear": 0.10,
    "disgust": 0.10
}

total = 0
for emotion, target_ratio in distribution.items():
    total += target_ratio
    current_count = df["Emotion_core"].value_counts().get(emotion, 0)
    current_ratio = current_count / len(df)
    print(f"Emotion: {emotion}, Current Ratio: {current_ratio:.2%}, Target Ratio: {target_ratio:.2%}")

print(total)

Emotion: neutral, Current Ratio: 61.58%, Target Ratio: 20.00%
Emotion: happiness, Current Ratio: 6.99%, Target Ratio: 16.50%
Emotion: surprise, Current Ratio: 16.63%, Target Ratio: 16.50%
Emotion: anger, Current Ratio: 3.59%, Target Ratio: 15.00%
Emotion: sadness, Current Ratio: 4.09%, Target Ratio: 12.00%
Emotion: fear, Current Ratio: 1.51%, Target Ratio: 10.00%
Emotion: disgust, Current Ratio: 5.61%, Target Ratio: 10.00%
1.0


In [11]:
def create_flexible_balanced_dataset(df, distribution, target_total_size=7000, flexibility=1.3):
    """
    Create a balanced dataset with flexibility to exceed target proportions

    Parameters:
    - flexibility: multiplier for how much over target is allowed (e.g., 1.3 = 30% over)
    """
    dfs = []
    actual_total = 0

    print("Target vs Flexible limits:")
    for emotion, target_ratio in distribution.items():
        target_count = int(target_ratio * target_total_size)
        flexible_limit = int(target_count * flexibility)
        available = df[df["Emotion_core"] == emotion].shape[0]

        # Take up to flexible limit, but not more than available
        take_count = min(flexible_limit, available)

        print(f"{emotion}: target={target_count}, flexible_limit={flexible_limit}, available={available}, taking={take_count}")

        if available >= take_count:
            df_emotion = df[df["Emotion_core"] == emotion].sample(take_count, random_state=42)
        else:
            df_emotion = df[df["Emotion_core"] == emotion]  # Take all available

        dfs.append(df_emotion)
        actual_total += len(df_emotion)

    df_balanced = pd.concat(dfs, ignore_index=True)

    print(f"\nActual total size: {actual_total}")
    print("\nFinal distribution:")
    for emotion in distribution.keys():
        count = (df_balanced["Emotion_core"] == emotion).sum()
        ratio = count / len(df_balanced)
        target_ratio = distribution[emotion]
        print(f"{emotion}: {count} ({ratio:.2%}) - target was {target_ratio:.2%}")

    return df_balanced

In [12]:
distribution = {
    "neutral": 0.20,
    "happiness": 0.165,
    "surprise": 0.165,
    "anger": 0.15,
    "sadness": 0.12,
    "fear": 0.10,
    "disgust": 0.10
}

df_balanced = create_flexible_balanced_dataset(
    df,
    distribution,
    target_total_size=7000,  # Increased base size
    flexibility=1.3  # Allow 30% over target
)

Target vs Flexible limits:
neutral: target=1400, flexible_limit=1820, available=15900, taking=1820
happiness: target=1155, flexible_limit=1501, available=1804, taking=1501
surprise: target=1155, flexible_limit=1501, available=4294, taking=1501
anger: target=1050, flexible_limit=1365, available=928, taking=928
sadness: target=840, flexible_limit=1092, available=1057, taking=1057
fear: target=700, flexible_limit=910, available=390, taking=390
disgust: target=700, flexible_limit=910, available=1448, taking=910

Actual total size: 8107

Final distribution:
neutral: 1820 (22.45%) - target was 20.00%
happiness: 1501 (18.51%) - target was 16.50%
surprise: 1501 (18.51%) - target was 16.50%
anger: 928 (11.45%) - target was 15.00%
sadness: 1057 (13.04%) - target was 12.00%
fear: 390 (4.81%) - target was 10.00%
disgust: 910 (11.22%) - target was 10.00%


In [14]:
# neutral capped at 25% rest between 12 and 17% show desired distribution
target_total_size = 7000
distribution = {
    "neutral": 0.20,
    "happiness": 0.165,
    "surprise": 0.165,
    "anger": 0.15,
    "sadness": 0.12,
    "fear": 0.10,
    "disgust": 0.10
}

total = 0
for emotion, target_ratio in distribution.items():
    total += target_ratio
    warning_flag = False
    current_count = df_balanced["Emotion_core"].value_counts().get(emotion, 0)
    current_ratio = current_count / len(df_balanced)
    if current_count < int(target_ratio * target_total_size):
        warning_flag = True
    print(f"Emotion: {emotion}, Current Ratio: {current_ratio:.2%}, Target Ratio: {target_ratio:.2%}. {'Warning: Undersampled' if warning_flag else ''}")

print(len(df_balanced))
print(total)

Emotion: neutral, Current Ratio: 22.45%, Target Ratio: 20.00%. 
Emotion: happiness, Current Ratio: 18.51%, Target Ratio: 16.50%. 
Emotion: surprise, Current Ratio: 18.51%, Target Ratio: 16.50%. 
Emotion: anger, Current Ratio: 11.45%, Target Ratio: 15.00%. Warning: Undersampled
Emotion: sadness, Current Ratio: 13.04%, Target Ratio: 12.00%. 
Emotion: fear, Current Ratio: 4.81%, Target Ratio: 10.00%. Warning: Undersampled
Emotion: disgust, Current Ratio: 11.22%, Target Ratio: 10.00%. 
8107
1.0


In [15]:
df_balanced["Emotion_core"].value_counts()

Emotion_core
neutral      1820
happiness    1501
surprise     1501
sadness      1057
anger         928
disgust       910
fear          390
Name: count, dtype: int64

In [16]:
df_balanced.to_csv("../../../data/dataset_balanced.csv", index=False)